<a href="https://colab.research.google.com/github/ShiftorTheOrca/asah-capstone/blob/main/src/Sistem_Rekomendasi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Library**


In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

# **Data Loading**


In [2]:
df = pd.read_csv('../assets/online_retail_uci_clustering.csv')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Category,Seasonality,Sales,Cluster,Label
0,538035,84341B,SMALL PINK MAGIC CHRISTMAS TREE,1.0,2010-12-09 13:03:00,0.85,16065.0,United Kingdom,christmas,Winter,0.85,2,Churn/One Time Spender
1,538035,22566,FELTCRAFT HAIRBAND PINK AND PURPLE,2.0,2010-12-09 13:03:00,0.85,16065.0,United Kingdom,others,Winter,1.70,2,Churn/One Time Spender
2,538035,22565,FELTCRAFT HAIRBANDS PINK AND WHITE,2.0,2010-12-09 13:03:00,0.85,16065.0,United Kingdom,others,Winter,1.70,2,Churn/One Time Spender
3,538035,22586,FELTCRAFT HAIRBAND PINK AND BLUE,2.0,2010-12-09 13:03:00,0.85,16065.0,United Kingdom,others,Winter,1.70,2,Churn/One Time Spender
4,538035,22587,FELTCRAFT HAIRBAND RED AND BLUE,2.0,2010-12-09 13:03:00,0.85,16065.0,United Kingdom,others,Winter,1.70,2,Churn/One Time Spender


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 318127 entries, 0 to 318126
Data columns (total 13 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    318127 non-null  int64  
 1   StockCode    318127 non-null  object 
 2   Description  318127 non-null  object 
 3   Quantity     318127 non-null  float64
 4   InvoiceDate  318127 non-null  object 
 5   UnitPrice    318127 non-null  float64
 6   CustomerID   318127 non-null  float64
 7   Country      318127 non-null  object 
 8   Category     318127 non-null  object 
 9   Seasonality  318127 non-null  object 
 10  Sales        318127 non-null  float64
 11  Cluster      318127 non-null  int64  
 12  Label        318127 non-null  object 
dtypes: float64(4), int64(2), object(7)
memory usage: 31.6+ MB


# **Feature Engineering**


Content Based Filtering

In [4]:
# Ambil setiap
items = df[['StockCode', 'Description', 'Category']]
items

,StockCode,Description,Category
0,84341B,SMALL PINK MAGIC CHRISTMAS TREE,christmas
1,22566,FELTCRAFT HAIRBAND PINK AND PURPLE,others
2,22565,FELTCRAFT HAIRBANDS PINK AND WHITE,others
3,22586,FELTCRAFT HAIRBAND PINK AND BLUE,others
4,22587,FELTCRAFT HAIRBAND RED AND BLUE,others
...,...,...,...
318122,23145,ZINC T-LIGHT HOLDER STAR LARGE,lighting
318123,22466,FAIRY TALE COTTAGE NIGHT LIGHT,lighting
318124,23275,SET OF 3 HANGING OWLS OLLIE BEAK,decoration
318125,21217,RED RETROSPOT ROUND CAKE TINS,storage


In [5]:
print("Data duplikat:", items.duplicated().sum())
items = items.drop_duplicates().reset_index().drop('index', axis=1)
items.duplicated().sum()

Data duplikat: 314323


np.int64(0)

In [6]:
items['Description'] = items['Description'].apply(lambda desc: desc.strip())
items['Tags'] = items['Description'] + " " + items['Category']
items['Tags'] = items['Tags'].apply(lambda x:x.lower().strip())
items

,StockCode,Description,Category,Tags
0,84341B,SMALL PINK MAGIC CHRISTMAS TREE,christmas,small pink magic christmas tree christmas
1,22566,FELTCRAFT HAIRBAND PINK AND PURPLE,others,feltcraft hairband pink and purple others
2,22565,FELTCRAFT HAIRBANDS PINK AND WHITE,others,feltcraft hairbands pink and white others
3,22586,FELTCRAFT HAIRBAND PINK AND BLUE,others,feltcraft hairband pink and blue others
4,22587,FELTCRAFT HAIRBAND RED AND BLUE,others,feltcraft hairband red and blue others
...,...,...,...,...
3799,90214Z,"LETTER ""Z"" BLING KEY RING",others,"letter ""z"" bling key ring others"
3800,90083,CRYSTAL CZECH CROSS PHONE CHARM,others,crystal czech cross phone charm others
3801,90089,PINK CRYSTAL SKULL PHONE CHARM,others,pink crystal skull phone charm others
3802,72783,BLACK SIL'T SQU CANDLE PLATE,kitchen,black sil't squ candle plate kitchen


# **Vectorizer**


In [7]:
cv = CountVectorizer(stop_words='english')
vector = cv.fit_transform(items['Tags']).toarray()

In [8]:
similarity = cosine_similarity(vector)

# **Inference**


In [9]:
def recommend_item_description(item_name_full, top_n=5):
  index = items[items['Description'] == item_name_full].index[0]
  distances = sorted(list(enumerate(similarity[index])),reverse=True,key = lambda x: x[1])
  
  print("Karena Anda membeli ", items.iloc[index]['Description'], ", mungkin Anda tertarik: ", sep="")
  
  for i in distances[1:top_n+1]:
    print(f"- {items.iloc[i[0]].Description} (Similarity: {i[1]:.4f})")

def recommend_item_code(item_code, top_n=5):
  index = items[items['StockCode'] == item_code].index[0]
  distances = sorted(list(enumerate(similarity[index])),reverse=True,key = lambda x: x[1])
  
  print("Karena Anda membeli ", items.iloc[index]['Description'], ", mungkin Anda tertarik: ", sep="")
  
  for i in distances[1:top_n+1]:
    print(f"- {items.iloc[i[0]].Description} (Similarity: {i[1]:.4f})")

In [10]:
recommend_item_description('LETTER "Z" BLING KEY RING')

Karena Anda membeli LETTER "Z" BLING KEY RING, mungkin Anda tertarik: 
- LETTER "J" BLING KEY RING (Similarity: 1.0000)
- LETTER "D" BLING KEY RING (Similarity: 1.0000)
- LETTER "K" BLING KEY RING (Similarity: 1.0000)
- LETTER "R" BLING KEY RING (Similarity: 1.0000)
- LETTER "G" BLING KEY RING (Similarity: 1.0000)


In [11]:
recommend_item_description(np.random.choice(items['Description'].values))

Karena Anda membeli BUTTONS AND STRIPES NOTEBOOK, mungkin Anda tertarik: 
- CHRYSANTHEMUM NOTEBOOK (Similarity: 0.5774)
- DAISY NOTEBOOK (Similarity: 0.5774)
- BLUE PAISLEY NOTEBOOK (Similarity: 0.5000)
- GARDEN PATH NOTEBOOK (Similarity: 0.5000)
- GREEN FERN NOTEBOOK (Similarity: 0.5000)


In [12]:
items.sample(5)

,StockCode,Description,Category,Tags
3594,23536,VILLAGE SHOW WALL ART,decoration,village show wall art decoration
1481,21557,SET OF 6 FUNKY BEAKERS,others,set of 6 funky beakers others
1146,22227,HANGING HEART MIRROR DECORATION,decoration,hanging heart mirror decoration decoration
1639,90129C,GREEN GLASS TASSLE BAG CHARM,bags,green glass tassle bag charm bags
484,22772,PINK DRAWER KNOB ACRYLIC EDWARDIAN,furniture,pink drawer knob acrylic edwardian furniture


In [13]:
recommend_item_code('23376')

Karena Anda membeli PACK OF 12 VINTAGE CHRISTMAS TISSUE, mungkin Anda tertarik: 
- PACK OF 12 50'S CHRISTMAS TISSUES (Similarity: 0.7500)
- PACK OF 12 CHRISTMAS FUN CARDS (Similarity: 0.7500)
- VINTAGE CHRISTMAS TABLECLOTH (Similarity: 0.7217)
- VINTAGE CHRISTMAS STOCKING (Similarity: 0.7217)
- PACK 3 BOXES CHRISTMAS PANNETONE (Similarity: 0.6682)


In [14]:
recommend_item_code(np.random.choice(items['StockCode'].values))

Karena Anda membeli SPACEBOY BEAKER, mungkin Anda tertarik: 
- MINI JIGSAW SPACEBOY (Similarity: 0.5774)
- TOY TIDY SPACEBOY (Similarity: 0.5774)
- KIDS CUTLERY SPACEBOY (Similarity: 0.5774)
- DOLLY GIRL BEAKER (Similarity: 0.5774)
- SPACEBOY MINI BACKPACK (Similarity: 0.5774)


In [15]:
sales_by_stockcode = df.groupby(['CustomerID', 'StockCode'])['Sales'].sum()
favorite_items = sales_by_stockcode.groupby('CustomerID').nlargest(5).droplevel(level=1).reset_index()

favorite_items

,CustomerID,StockCode,Sales
0,12747.0,82484,1319.40
1,12747.0,82482,325.20
2,12747.0,82494L,315.60
3,12747.0,85062,208.20
4,12747.0,84879,148.72
...,...,...,...
18530,18287.0,85039B,176.40
18531,18287.0,85039A,139.20
18532,18287.0,72349B,106.32
18533,18287.0,85173,81.12


In [22]:
customer_dict = favorite_items.groupby('CustomerID')['StockCode'].apply(list).to_dict()

def recommend_for_single_customer(idcustomer, top_n=5):
    if idcustomer not in customer_dict:
        print(f"CustomerID {idcustomer} tidak ditemukan")
        return

    print(f"CustomerID: {idcustomer}")

    stock_codes = customer_dict[idcustomer]
    all_similarities = []

    for stock_code in stock_codes:
        index = items[items['StockCode'] == stock_code].index[0]
        distances = sorted(list(enumerate(similarity[index])))
        all_similarities.extend(distances)

        print(f"- Anda membeli {items.iloc[index]['Description']}")

    similarity_dict = {}

    for idx, sim in all_similarities:
        if idx in similarity_dict:
            similarity_dict[idx].append(sim)
        else:
            similarity_dict[idx] = [sim]

    avg_similarities = []

    for idx, sims in similarity_dict.items():
        if items.iloc[idx]['StockCode'] not in stock_codes:
            avg_similarities.append((idx, np.mean(sims)))

    avg_similarities = sorted(avg_similarities,reverse=True, key = lambda x: x[1])
    recommendations = []

    for i in range(top_n):
        idx, sim_score = avg_similarities[i]
        recommendations.append({
            'StockCode': items.iloc[idx]['StockCode'],
            'Description': items.iloc[idx]['Description'],
            'Similarity': sim_score
        })

    print("\nRekomendasi untuk customer ini:")

    for i, rec in enumerate(recommendations):
        print(f"{i + 1}. {rec['Description']} (Similarity: {rec['Similarity']:.4f})")

In [23]:
recommend_for_single_customer(12748)

CustomerID: 12748
- Anda membeli LAZER CUT NECKLACE W PASTEL BEADS
- Anda membeli SILVER FLOWR PINK SHELL NECKLACE
- Anda membeli REGENCY TEAPOT ROSES
- Anda membeli ROSE SCENT CANDLE JEWELLED DRAWER
- Anda membeli VANILLA SCENT CANDLE JEWELLED BOX

Rekomendasi untuk customer ini:
1. ROSE SCENT CANDLE IN JEWELLED BOX (Similarity: 0.3333)
2. OCEAN SCENT CANDLE IN JEWELLED BOX (Similarity: 0.3000)
3. SET/3 ROSE CANDLE IN JEWELLED BOX (Similarity: 0.2667)
4. COFFEE SCENT PILLAR CANDLE (Similarity: 0.2191)
5. LAVENDER SCENT CAKE CANDLE (Similarity: 0.2191)


In [24]:
recommend_for_single_customer(np.random.choice(list(customer_dict.keys())))

CustomerID: 14867.0
- Anda membeli ASSORTED COLOUR BIRD ORNAMENT
- Anda membeli 6 EGG HOUSE PAINTED WOOD
- Anda membeli VINTAGE CREAM 3 BASKET CAKE STAND
- Anda membeli WOOD BLACK BOARD ANT WHITE FINISH
- Anda membeli LADIES & GENTLEMEN METAL SIGN

Rekomendasi untuk customer ini:
1. 12 EGG HOUSE PAINTED WOOD (Similarity: 0.2154)
2. HOUSE WRECKING METAL SIGN (Similarity: 0.2047)
3. DARK BIRD HOUSE TREE DECORATION (Similarity: 0.1935)
4. HEN HOUSE DECORATION (Similarity: 0.1869)
5. SWEETHEART BIRD HOUSE (Similarity: 0.1842)
